In [ ]:
import sys

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q opencv-python-headless
!pip install -q sacremoses
!pip install -q dtw-python
!pip install -q einops
!pip install -q timm
!pip install -q openai-whisper

In [ ]:
# ── Fix version conflicts ────────────────────────────────────────
!pip install -q sentencepiece
!pip install -q "transformers==4.44.0"
!pip install -q "huggingface_hub==0.24.0"
!pip install -q "tokenizers==0.19.1"

In [ ]:
!pip uninstall mediapipe -y

!pip install mediapipe==0.10.14

### Imports + Mount Drive

In [ ]:
import os, sys, json, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
import mediapipe as mp
from google.colab import drive, output

# Transformers
from transformers import (
    MarianMTModel, MarianTokenizer,     # EN→DE
    AutoTokenizer, AutoModel            # gloss embeddings
)

# Swin Transformer (from timm)
import timm

# Progressive Transformer path
sys.path.insert(0, "/content/ProgressiveTransformersSLP")

# Mount Google Drive
drive.mount("/content/drive")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {DEVICE}")
print(f"   CUDA available: {torch.cuda.is_available()}")

### Paths

In [ ]:
import os, shutil

SRC_BASE = '/content/drive/MyDrive/Final Year Project - 2026/Phoenix'

# ── Destination (Colab local — faster I/O than Drive) ──
DST_BASE = '/content/PHOENIX14T'

# Copy dataset to local Colab storage if not already done
if not os.path.exists(DST_BASE):
    print("📂 Copying dataset from Drive to local Colab... (one-time, ~few mins)")
    shutil.copytree(SRC_BASE, DST_BASE)
    print("✅ Dataset copied.")
else:
    print("✅ Dataset already local.")

# ── Video directories ──
TRAIN_VIDEO_DIR = os.path.join(DST_BASE, 'videos_phoenix/videos/train')
DEV_VIDEO_DIR   = os.path.join(DST_BASE, 'videos_phoenix/videos/dev')
TEST_VIDEO_DIR  = os.path.join(DST_BASE, 'videos_phoenix/videos/test')

# ── Annotation files (.gzip) ──
TRAIN_ANN = os.path.join(DST_BASE, 'phoenix14t.pami0.train.annotations_only.gzip')
DEV_ANN   = os.path.join(DST_BASE, 'phoenix14t.pami0.dev.annotations_only.gzip')
TEST_ANN  = os.path.join(DST_BASE, 'phoenix14t.pami0.test.annotations_only.gzip')

# ── Project output dirs (keep on Drive so they survive session resets) ──
BASE       = "/content/drive/MyDrive/DGS_Project"
MODEL_DIR  = os.path.join(BASE, "models")
OUTPUT_DIR = os.path.join(BASE, "outputs")
CKPT_DIR   = os.path.join(BASE, "checkpoints")

for d in [MODEL_DIR, OUTPUT_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Keypoints dir (local — large files, rebuild each session or save to Drive) ──
PHOENIX_KEYPTS = os.path.join(DST_BASE, 'keypoints')
os.makedirs(PHOENIX_KEYPTS, exist_ok=True)

# ── Keypoint dimensions ──
N_BODY_KPS  = 33    # MediaPipe Pose
N_HAND_KPS  = 21    # each hand (42 total)
N_FACE_LMS  = 468   # MediaPipe Face Mesh
N_TOTAL_KPS = N_BODY_KPS + 42 + N_FACE_LMS   # 543
COORDS      = 3

# ── Verify structure ──
print("\n📁 Directory check:")
for label, path in [
    ("Train videos", TRAIN_VIDEO_DIR),
    ("Dev videos",   DEV_VIDEO_DIR),
    ("Test videos",  TEST_VIDEO_DIR),
]:
    if os.path.exists(path):
        count = len(os.listdir(path))
        print(f"   {label}: {count} items")
    else:
        print(f"   ❌ NOT FOUND: {path}")

print("\n📄 Annotation files:")
for label, path in [
    ("Train", TRAIN_ANN),
    ("Dev",   DEV_ANN),
    ("Test",  TEST_ANN),
]:
    exists = "✅" if os.path.exists(path) else "❌ NOT FOUND"
    print(f"   {exists}  {label}: {path}")

print(f"\n✅ Paths configured.")
print(f"   Total keypoints per frame: {N_TOTAL_KPS} × {COORDS} = {N_TOTAL_KPS * COORDS} dims")

In [ ]:
# Replace in imports cell
from transformers import (
    MarianMTModel, MarianTokenizer,
    BertTokenizer,                    # ← replace AutoTokenizer with this
    AutoModel
)

### Whisper

In [ ]:

import whisper

_whisper_model = None

def load_whisper():
    global _whisper_model
    if _whisper_model is None:
        _whisper_model = whisper.load_model("medium")
        print("✅ Whisper loaded (medium).")
    return _whisper_model

def transcribe_to_german(audio_path: str) -> str:
    """
    Transcribe audio file → German text using Whisper.
    """
    model = load_whisper()
    result = model.transcribe(audio_path, language="de", task="transcribe")
    text = result["text"].strip()
    print(f"📝 Whisper output: {text}")
    return text

### TextGlossDataset

In [ ]:
import gzip, pickle

def load_phoenix_annotations(gzip_path):
    """Load PHOENIX-2014T .gzip annotation file → list of dicts."""
    with gzip.open(gzip_path, 'rb') as f:
        data = pickle.load(f)
    # Each entry has keys: 'name', 'gloss', 'text', 'sign' (video frames path)
    return data

# Load all splits
train_annotations = load_phoenix_annotations(TRAIN_ANN)
dev_annotations   = load_phoenix_annotations(DEV_ANN)
test_annotations  = load_phoenix_annotations(TEST_ANN)

print(f"✅ Annotations loaded:")
print(f"   Train: {len(train_annotations)} samples")
print(f"   Dev:   {len(dev_annotations)} samples")
print(f"   Test:  {len(test_annotations)} samples")

# Preview one sample
sample = train_annotations[0]
print(f"\n📋 Sample keys: {list(sample.keys())}")
print(f"   name:  {sample['name']}")
print(f"   gloss: {sample['gloss']}")
print(f"   text:  {sample['text']}")

In [ ]:
import csv

class PHOENIXGlossDataset(Dataset):
    def __init__(self, annotations, keypoint_dir, max_seq_len=64, max_gloss_len=50):
        self.samples       = annotations
        self.keypoint_dir  = keypoint_dir
        self.max_seq_len   = max_seq_len
        self.max_gloss_len = max_gloss_len
        print(f"✅ Dataset: {len(self.samples)} samples | max_seq_len={max_seq_len}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s         = self.samples[idx]
        safe_name = s['name'].replace('/', '_')
        kp_path   = os.path.join(self.keypoint_dir, safe_name + ".npy")

        if os.path.exists(kp_path):
            kps = np.load(kp_path)
        else:
            kps = np.zeros((30, N_TOTAL_KPS, 3), dtype=np.float32)

        T = kps.shape[0]
        if T > self.max_seq_len:
            kps = kps[:self.max_seq_len]
        elif T < self.max_seq_len:
            pad = np.zeros((self.max_seq_len - T, N_TOTAL_KPS, 3), dtype=np.float32)
            kps = np.concatenate([kps, pad], axis=0)

        body_kps    = kps[:, :N_BODY_KPS + 42, :]
        face_kps    = kps[:, N_BODY_KPS + 42:, :]
        gloss_field = s.get('gloss') or s.get('glosses') or ''
        if isinstance(gloss_field, str):
            gloss_field = gloss_field.split()

        return {
            "name":     safe_name,
            "text":     s.get('text', ''),
            "glosses":  gloss_field[:self.max_gloss_len],
            "body_kps": torch.tensor(body_kps, dtype=torch.float32),
            "face_kps": torch.tensor(face_kps, dtype=torch.float32),
            "full_kps": torch.tensor(kps,      dtype=torch.float32),
        }

def build_gloss_vocab(dataset):
    vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
    for s in dataset.samples:
        gloss_field = s.get("gloss") or s.get("glosses") or ""
        if isinstance(gloss_field, str):
            gloss_field = gloss_field.split()
        for g in gloss_field:
            if g not in vocab:
                vocab[g] = len(vocab)
    return vocab

### DE→Gloss Model

In [ ]:
# ── Run this FIRST before Cell 6 ──
train_ds = PHOENIXGlossDataset(train_annotations, PHOENIX_KEYPTS)
gloss_vocab = build_gloss_vocab(train_ds)

print(f"✅ Vocab size: {len(gloss_vocab)}")

In [ ]:
# ── Data + Vocab + Loaders + Model ──────────────────────────────
train_annotations = load_phoenix_annotations(TRAIN_ANN)
dev_annotations   = load_phoenix_annotations(DEV_ANN)

train_ds    = PHOENIXGlossDataset(train_annotations, PHOENIX_KEYPTS)
val_ds      = PHOENIXGlossDataset(dev_annotations,   PHOENIX_KEYPTS)
gloss_vocab = build_gloss_vocab(train_ds)
print(f"✅ Vocab size: {len(gloss_vocab)}")

In [ ]:
class DEGlossModel(nn.Module):
    """
    German Text → DGS Gloss sequence.
    Uses mBART / mT5 encoder + custom gloss decoder.
    For paper: replace with your trained checkpoint.
    """

    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # Text encoder (pretrained German BERT)
        from transformers import AutoModel
        self.text_encoder = AutoModel.from_pretrained("deepset/gbert-base")
        text_dim = 768

        # Project BERT hidden → d_model
        self.text_proj = nn.Linear(text_dim, d_model)

        # Gloss embedding
        self.gloss_embed = nn.Embedding(vocab_size, d_model, padding_idx=0)

        # Transformer decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        # Output projection → vocab
        self.out_proj = nn.Linear(d_model, vocab_size)
        self.pos_enc  = self._positional_encoding(512, d_model)

    def _positional_encoding(self, max_len, d_model):
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe.unsqueeze(0)  # (1, max_len, d_model)

    def encode_text(self, input_ids, attention_mask):
        with torch.no_grad():
            bert_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        enc = self.text_proj(bert_out.last_hidden_state)  # (B, L, d_model)
        return enc

    def forward(self, input_ids, attention_mask, gloss_ids):
        B, T = gloss_ids.shape
        memory = self.encode_text(input_ids, attention_mask)

        gloss_emb = self.gloss_embed(gloss_ids)
        pe = self.pos_enc[:, :T, :].to(gloss_emb.device)
        gloss_emb = gloss_emb + pe

        out = self.decoder(tgt=gloss_emb, memory=memory)
        logits = self.out_proj(out)  # (B, T, vocab_size)
        return logits

    @torch.no_grad()
    def generate(self, text, tokenizer, gloss_vocab, max_len=50, device="cpu"):
        """Greedy decoding: German text → gloss list."""
        self.eval()
        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
        memory = self.encode_text(enc["input_ids"], enc["attention_mask"])

        inv_vocab = {v: k for k, v in gloss_vocab.items()}
        sos_id = gloss_vocab["<sos>"]
        eos_id = gloss_vocab["<eos>"]

        tokens = [sos_id]
        for _ in range(max_len):
            ids = torch.tensor([tokens], device=device)
            gloss_emb = self.gloss_embed(ids)
            pe = self.pos_enc[:, :len(tokens), :].to(device)
            gloss_emb = gloss_emb + pe
            out = self.decoder(tgt=gloss_emb, memory=memory)
            logits = self.out_proj(out[:, -1, :])
            next_id = logits.argmax(-1).item()
            if next_id == eos_id:
                break
            tokens.append(next_id)

        return [inv_vocab.get(t, "<unk>") for t in tokens[1:]]


# Load (or init) DE→Gloss model
from transformers import AutoTokenizer

de_gloss_tokenizer = BertTokenizer.from_pretrained("deepset/gbert-base")

de_gloss_model = DEGlossModel(vocab_size=len(gloss_vocab)).to(DEVICE)
ckpt = os.path.join(CKPT_DIR, "de_gloss_model.pt")
if os.path.exists(ckpt):
     de_gloss_model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
     print("✅ DE→Gloss model loaded from checkpoint.")

print("✅ DE→Gloss model defined.")


### EN→DE Translation Model

In [ ]:

_en_de_model = None
_en_de_tokenizer = None

def load_en_de():
    global _en_de_model, _en_de_tokenizer
    if _en_de_model is None:
        model_name = "Helsinki-NLP/opus-mt-en-de"
        _en_de_tokenizer = MarianTokenizer.from_pretrained(model_name)
        _en_de_model = MarianMTModel.from_pretrained(model_name).to(DEVICE)
        _en_de_model.eval()
        print("✅ EN→DE MarianMT loaded.")
    return _en_de_model, _en_de_tokenizer

def translate_en_to_de(english_text: str) -> str:
    model, tokenizer = load_en_de()
    inputs = tokenizer([english_text], return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        translated = model.generate(**inputs, num_beams=4, max_length=512)
    german = tokenizer.decode(translated[0], skip_special_tokens=True)
    print(f"🇩🇪 German: {german}")
    return german


# Example:
de_text = translate_en_to_de("Good morning, how are you?")

### DUAL-STREAM MODEL

In [ ]:
#
# Architecture:
#
#   Gloss Tokens
#        │
#   ┌────┴───────────────────────────┐
#   │  Stream 1: Body/Hand Generator │  ← Progressive Transformer backbone
#   │  Stream 2: Face Generator      │  ← Swin Transformer (video-pretrained)
#   └────┬──────────────┬────────────┘
#        │   Cross-Attention Fusion   │  ← YOUR NOVEL COMPONENT
#        └──────────────┘
#              │
#   Full Keypoints (body + face)
#
# ─────────────────────────────────────────

### a. Keypoint Extractor (MediaPipe)

In [ ]:
class KeypointExtractor:
    """
    Extract 543 keypoints per frame from PHOENIX videos.
    Body(33) + Hands(21×2) + Face(468) = 543 landmarks.
    """

    def __init__(self):
        self.mp_pose      = mp.solutions.pose
        self.mp_hands     = mp.solutions.hands
        self.mp_face_mesh = mp.solutions.face_mesh

        self.pose      = self.mp_pose.Pose(static_image_mode=False, model_complexity=2)
        self.hands     = self.mp_hands.Hands(static_image_mode=False, max_num_hands=2)
        self.face_mesh = self.mp_face_mesh.FaceMesh(static_image_mode=False, refine_landmarks=True)

    def extract_frame(self, frame_rgb):
        """Extract keypoints from a single RGB frame. Returns (543, 3)."""
        kps = np.zeros((N_TOTAL_KPS, 3), dtype=np.float32)

        # Body (33 landmarks)
        pose_res = self.pose.process(frame_rgb)
        if pose_res.pose_landmarks:
            for i, lm in enumerate(pose_res.pose_landmarks.landmark):
                kps[i] = [lm.x, lm.y, lm.z]

        # Hands (21 × 2 = 42 landmarks)
        hand_res = self.hands.process(frame_rgb)
        left_written, right_written = False, False
        if hand_res.multi_hand_landmarks and hand_res.multi_handedness:
            for hand_lms, handedness in zip(
                hand_res.multi_hand_landmarks, hand_res.multi_handedness
            ):
                label = handedness.classification[0].label
                offset = N_BODY_KPS if label == "Left" else N_BODY_KPS + 21
                for i, lm in enumerate(hand_lms.landmark):
                    kps[offset + i] = [lm.x, lm.y, lm.z]

        # Face Mesh (468 landmarks)
        face_res = self.face_mesh.process(frame_rgb)
        offset = N_BODY_KPS + 42
        if face_res.multi_face_landmarks:
            for i, lm in enumerate(face_res.multi_face_landmarks[0].landmark):
                if i >= N_FACE_LMS:
                    break
                kps[offset + i] = [lm.x, lm.y, lm.z]

        return kps

    def extract_video(self, video_path, save_path=None):
        """Extract all frames from a video. Returns (T, 543, 3)."""
        cap = cv2.VideoCapture(video_path)
        frames_kps = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            kps = self.extract_frame(rgb)
            frames_kps.append(kps)
        cap.release()

        result = np.array(frames_kps, dtype=np.float32)
        if save_path:
            np.save(save_path, result)
            print(f"💾 Saved keypoints: {result.shape} → {save_path}")
        return result

    def extract_dataset(self, video_dir, out_dir):
        """Batch-extract keypoints for all PHOENIX videos."""
        os.makedirs(out_dir, exist_ok=True)
        videos = [f for f in os.listdir(video_dir) if f.endswith(".mp4")]
        print(f"Extracting keypoints for {len(videos)} videos...")
        for i, vid in enumerate(videos):
            name = vid.replace(".mp4", "")
            out_path = os.path.join(out_dir, name + ".npy")
            if not os.path.exists(out_path):
                self.extract_video(os.path.join(video_dir, vid), out_path)
            if (i + 1) % 50 == 0:
                print(f"  {i+1}/{len(videos)} done")
        print("✅ Keypoint extraction complete.")


### b. Body Stream: Progressive Transformer Backbone

In [ ]:
class BodyStreamTransformer(nn.Module):
    """
    Stream 1: Gloss → Body + Hand keypoints.
    Based on Progressive Transformer (Saunders et al., 2020).
    Modified to output embeddings for cross-attention.
    """

    def __init__(self, gloss_vocab_size, d_model=256, nhead=4,
                 num_encoder_layers=2, num_decoder_layers=2,
                 body_dim=75*3, dropout=0.1):
        super().__init__()
        self.d_model  = d_model
        self.body_dim = body_dim
        self.gloss_embed = nn.Embedding(gloss_vocab_size, d_model, padding_idx=0)
        self.pos_enc     = self._build_pos_enc(512, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True)
        self.gloss_encoder = nn.TransformerEncoder(enc_layer, num_layers=num_encoder_layers)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True)
        self.pose_decoder    = nn.TransformerDecoder(dec_layer, num_layers=num_decoder_layers)
        self.pose_input_proj = nn.Linear(body_dim, d_model)
        self.pose_out        = nn.Linear(d_model, body_dim)
        self.count_out       = nn.Linear(d_model, 1)

    def _build_pos_enc(self, max_len, d_model):
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe.unsqueeze(0)

    def encode_glosses(self, gloss_ids):
        B, G = gloss_ids.shape
        emb  = self.gloss_embed(gloss_ids)
        pe   = self.pos_enc[:, :G, :].to(emb.device)
        return self.gloss_encoder(emb + pe)

    def forward(self, gloss_ids, body_kps_in):
        memory   = self.encode_glosses(gloss_ids)
        B, T, _  = body_kps_in.shape
        pose_emb = self.pose_input_proj(body_kps_in)
        pe       = self.pos_enc[:, :T, :].to(pose_emb.device)
        pose_emb = pose_emb + pe
        mask     = nn.Transformer.generate_square_subsequent_mask(T).to(pose_emb.device)
        body_h   = self.pose_decoder(tgt=pose_emb, memory=memory, tgt_mask=mask)
        return self.pose_out(body_h), self.count_out(body_h), body_h


### c. Face Stream: Swin Transformer

In [ ]:
class FaceStreamSwin(nn.Module):
    def __init__(self, gloss_vocab_size, d_model=256, nhead=4,
                 num_decoder_layers=2, face_dim=468*3, dropout=0.1):
        super().__init__()
        self.d_model  = d_model
        self.face_dim = face_dim

        # Lightweight MLP — NO Swin, NO 150528 projection
        self.face_landmark_proj = nn.Sequential(
            nn.Linear(face_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Linear(512, d_model),
            nn.LayerNorm(d_model),
        )
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model*2,
            dropout=dropout, batch_first=True)
        self.temporal_encoder = nn.TransformerEncoder(enc_layer, num_layers=1)

        self.gloss_embed  = nn.Embedding(gloss_vocab_size, d_model, padding_idx=0)
        self.pos_enc      = self._build_pos_enc(512, d_model)
        enc_layer2 = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True)
        self.gloss_encoder   = nn.TransformerEncoder(enc_layer2, num_layers=2)
        self.face_input_proj = nn.Linear(face_dim, d_model)

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True)
        self.face_decoder = nn.TransformerDecoder(dec_layer, num_layers=num_decoder_layers)
        self.face_out     = nn.Linear(d_model, face_dim)
        self.au_out       = nn.Linear(d_model, 12)

    def _build_pos_enc(self, max_len, d_model):
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe.unsqueeze(0)

    def encode_glosses(self, gloss_ids):
        emb = self.gloss_embed(gloss_ids)
        pe  = self.pos_enc[:, :gloss_ids.shape[1], :].to(emb.device)
        return self.gloss_encoder(emb + pe)

    def extract_swin_features(self, face_seq):
        B, T, D = face_seq.shape
        proj = self.face_landmark_proj(face_seq.reshape(B*T, D))
        return self.temporal_encoder(proj.reshape(B, T, -1))

    def forward(self, gloss_ids, face_kps_in):
        memory   = self.encode_glosses(gloss_ids)
        B, T, _  = face_kps_in.shape
        face_emb = self.face_input_proj(face_kps_in)
        pe       = self.pos_enc[:, :T, :].to(face_emb.device)
        face_emb = face_emb + pe
        mask     = nn.Transformer.generate_square_subsequent_mask(T).to(face_emb.device)
        face_h   = self.face_decoder(tgt=face_emb, memory=memory, tgt_mask=mask)
        return self.face_out(face_h), self.au_out(face_h), face_h



### d. Cross-Attention Fusion

In [ ]:
class CrossAttentionFusion(nn.Module):
    """
    *** CORE NOVEL COMPONENT ***

    Fuses body stream and face stream outputs via bidirectional
    cross-attention, allowing each stream to condition on the other.

    Body stream attends to face → hand shapes consistent with expression.
    Face stream attends to body → expression consistent with gesture.

    This is the main architectural novelty for the paper.
    """

    def __init__(self, d_model=256, nhead=4, num_layers=1, dropout=0.1):
        super().__init__()
        self.body_to_face_attn = nn.ModuleList([
            nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
            for _ in range(num_layers)])
        self.face_to_body_attn = nn.ModuleList([
            nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
            for _ in range(num_layers)])
        self.body_norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)])
        self.face_norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)])
        self.body_ffn   = nn.Sequential(nn.Linear(d_model, d_model*4), nn.GELU(), nn.Linear(d_model*4, d_model))
        self.face_ffn   = nn.Sequential(nn.Linear(d_model, d_model*4), nn.GELU(), nn.Linear(d_model*4, d_model))
        self.body_ffn_norm  = nn.LayerNorm(d_model)
        self.face_ffn_norm  = nn.LayerNorm(d_model)
        self.final_body_proj = nn.Linear(d_model, 75*3)
        self.final_face_proj = nn.Linear(d_model, 468*3)

    def forward(self, body_hidden, face_hidden):
        body_h = body_hidden
        face_h = face_hidden
        for i in range(len(self.body_to_face_attn)):
            body_attn, _ = self.body_to_face_attn[i](query=body_h, key=face_h, value=face_h)
            body_h = self.body_norms[i](body_h + body_attn)
            face_attn, _ = self.face_to_body_attn[i](query=face_h, key=body_h, value=body_h)
            face_h = self.face_norms[i](face_h + face_attn)
        body_h = self.body_ffn_norm(body_h + self.body_ffn(body_h))
        face_h = self.face_ffn_norm(face_h + self.face_ffn(face_h))
        return self.final_body_proj(body_h), self.final_face_proj(face_h)

### e. Complete Dual-Stream Model

In [ ]:
class DualStreamSignProducer(nn.Module):
    """
    Complete Option 1 architecture:
    Gloss → [Body Stream + Face Stream] → Cross-Attention → Full Pose
    """

    def __init__(self, gloss_vocab_size, d_model=256):
        super().__init__()
        self.body_stream = BodyStreamTransformer(gloss_vocab_size, d_model=d_model, nhead=4,
                                                  num_encoder_layers=2, num_decoder_layers=2)
        self.face_stream = FaceStreamSwin(gloss_vocab_size, d_model=d_model, nhead=4,
                                           num_decoder_layers=2)
        self.fusion      = CrossAttentionFusion(d_model=d_model, nhead=4, num_layers=1)

    def forward(self, gloss_ids, body_kps_in, face_kps_in):
        body_pred, counter_pred, body_h = self.body_stream(gloss_ids, body_kps_in)
        face_pred, au_pred,      face_h = self.face_stream(gloss_ids, face_kps_in)
        final_body, final_face          = self.fusion(body_h, face_h)
        return {
            "body_kps":     final_body,
            "face_kps":     final_face,
            "body_kps_raw": body_pred,
            "face_kps_raw": face_pred,
            "counter":      counter_pred,
            "au_pred":      au_pred,
        }

    @torch.no_grad()
    def generate(self, gloss_ids, max_frames=150, device="cpu"):
        """
        Auto-regressive generation (inference).
        Returns: (T, 543, 3) full keypoint sequence
        """
        self.eval()
        gloss_ids = gloss_ids.to(device)
        B = gloss_ids.shape[0]

        # Encode glosses
        body_memory = self.body_stream.encode_glosses(gloss_ids)
        face_memory = self.face_stream.encode_glosses(gloss_ids)

        body_seq = torch.zeros(B, 1, 75 * 3, device=device)
        face_seq = torch.zeros(B, 1, 468 * 3, device=device)

        generated_body = []
        generated_face = []

        for t in range(max_frames):
            # Body step
            body_emb = self.body_stream.pose_input_proj(body_seq)
            T_cur = body_seq.shape[1]
            pe = self.body_stream.pos_enc[:, :T_cur, :].to(device)
            body_emb = body_emb + pe
            mask = nn.Transformer.generate_square_subsequent_mask(T_cur).to(device)
            body_h = self.body_stream.pose_decoder(tgt=body_emb, memory=body_memory, tgt_mask=mask)
            body_step = self.body_stream.pose_out(body_h[:, -1:, :])
            counter   = self.body_stream.count_out(body_h[:, -1:, :])

            # Face step
            swin_ft  = self.face_stream.extract_swin_features(face_seq)
            face_emb = self.face_stream.face_input_proj(face_seq) + swin_ft + pe[:, :T_cur, :]
            face_h   = self.face_stream.face_decoder(tgt=face_emb, memory=face_memory, tgt_mask=mask)
            face_step = self.face_stream.face_out(face_h[:, -1:, :])
            # Cross-attention fusion for this step
            fused_body, fused_face = self.fusion(body_h[:, -1:, :], face_h[:, -1:, :])

            generated_body.append(fused_body.squeeze(1))
            generated_face.append(fused_face.squeeze(1))

            # Stop if counter decays to 0 (simplified)
            if t > 5 and counter.mean().item() < 0.1:
                break

            body_seq = torch.cat([body_seq, body_step], dim=1)
            face_seq = torch.cat([face_seq, face_step], dim=1)

        body_out = torch.stack(generated_body, dim=1)  # (B, T, 225)
        face_out = torch.stack(generated_face, dim=1)  # (B, T, 1404)

        # Combine to full keypoints (B, T, 543, 3)
        T = body_out.shape[1]
        full = torch.zeros(B, T, N_TOTAL_KPS, 3, device=device)
        full[:, :, :75,  :] = body_out.reshape(B, T, 75,  3)
        full[:, :, 75:,  :] = face_out.reshape(B, T, 468, 3)

        return full[0].cpu().numpy()  # (T, 543, 3)


### f. Loss Functions

In [ ]:
class DualStreamLoss(nn.Module):
    """
    Combined loss for paper.
    L_total = λ1·L_body + λ2·L_face + λ3·L_counter + λ4·L_au + λ5·L_smooth
    """

    def __init__(self, l1=1.0, l2=1.5, l3=0.5, l4=0.3, l5=0.2):
        super().__init__()
        self.l1, self.l2, self.l3, self.l4, self.l5 = l1, l2, l3, l4, l5

    def smoothness_loss(self, kps):
        vel   = kps[:, 1:] - kps[:, :-1]
        accel = vel[:, 1:] - vel[:, :-1]
        return accel.pow(2).mean()

    def forward(self, pred, target_body, target_face, target_counter, target_au=None):
        L_body    = F.mse_loss(pred["body_kps"], target_body)
        L_face    = F.mse_loss(pred["face_kps"], target_face)
        L_counter = F.mse_loss(pred["counter"].squeeze(-1), target_counter)
        L_smooth  = self.smoothness_loss(pred["body_kps"]) + self.smoothness_loss(pred["face_kps"])
        L_au      = torch.tensor(0.0, device=target_body.device)
        if target_au is not None:
            L_au  = F.binary_cross_entropy_with_logits(pred["au_pred"], target_au.float())
        total = self.l1*L_body + self.l2*L_face + self.l3*L_counter + self.l4*L_au + self.l5*L_smooth
        return total, {
            "body": L_body.item(), "face": L_face.item(),
            "counter": L_counter.item(), "au": L_au.item(),
            "smooth": L_smooth.item(), "total": total.item()
        }


### g. Training Loop

In [ ]:
# ── Step 1: Find the correct filename pattern ────────────────────

sample_name = train_annotations[0]['name']
# annotation: "train/11August_2010_Wednesday_tagesschau-1"
# actual file: "11August_2010_Wednesday_tagesschau-6694.mp4"

# Extract base name without the index suffix
base = sample_name.split('/')[-1]          # "11August_2010_Wednesday_tagesschau-1"
base_nonum = '-'.join(base.split('-')[:-1]) # "11August_2010_Wednesday_tagesschau"
print(f"Base name     : {base}")
print(f"Base no-num   : {base_nonum}")

# Find all mp4s that start with this base
matches = [
    f for f in os.listdir(TRAIN_VIDEO_DIR)
    if f.startswith(base_nonum) and f.endswith('.mp4')
]
print(f"Matching files: {matches}")
print()

# Check a few more annotations to understand the pattern
for ann in train_annotations[:5]:
    name     = ann['name'].split('/')[-1]
    base_nn  = '-'.join(name.split('-')[:-1])
    idx      = name.split('-')[-1]           # the annotation index
    matches  = [
        f for f in os.listdir(TRAIN_VIDEO_DIR)
        if f.startswith(base_nn) and f.endswith('.mp4')
    ]
    print(f"Ann  : {name}")
    print(f"Index: {idx}  →  matches: {matches[:3]}")
    print()

In [ ]:
# ── Step 2: Build annotation → mp4 path lookup ──────────────────

# Build index: base_name → sorted list of mp4s
from collections import defaultdict

mp4_index = defaultdict(list)
for fname in sorted(os.listdir(TRAIN_VIDEO_DIR)):
    if fname.endswith('.mp4'):
        base = '-'.join(fname.replace('.mp4','').split('-')[:-1])
        mp4_index[base].append(fname)

print(f"Unique base names in train: {len(mp4_index)}")
print(f"Example: {list(mp4_index.items())[:2]}")

In [ ]:
def get_video_path(annotation_name, video_dir, mp4_index):
    """
    Map annotation name → actual .mp4 path.
    annotation_name: "train/11August_2010_Wednesday_tagesschau-1"
    """
    base_full = annotation_name.split('/')[-1]
    # e.g. "11August_2010_Wednesday_tagesschau-1"

    base_nonum = '-'.join(base_full.split('-')[:-1])
    # e.g. "11August_2010_Wednesday_tagesschau"

    ann_idx = int(base_full.split('-')[-1])
    # e.g. 1  (0-based or 1-based index into matching mp4s)

    candidates = mp4_index.get(base_nonum, [])
    if not candidates:
        return None

    # Try as 1-based index into sorted list
    idx = ann_idx - 1
    if 0 <= idx < len(candidates):
        return os.path.join(video_dir, candidates[idx])

    # Fallback: return first match
    return os.path.join(video_dir, candidates[0])


# ── Verify mapping on first 5 samples ───────────────────────────
print("Verifying path mapping:")
for ann in train_annotations[:5]:
    path = get_video_path(ann['name'], TRAIN_VIDEO_DIR, mp4_index)
    exists = os.path.exists(path) if path else False
    print(f"  {ann['name']:<50} → {os.path.basename(path) if path else 'NOT FOUND'} {'✅' if exists else '❌'}")

In [ ]:
import os

PHOENIX_KEYPTS = "/content/drive/MyDrive/DGS_Project/keypoints"
npy_files = [f for f in os.listdir(PHOENIX_KEYPTS) if f.endswith('.npy')]
train_done = [f for f in npy_files if f.startswith('train_')]
dev_done   = [f for f in npy_files if f.startswith('dev_')]

print(f"Total saved    : {len(npy_files)}")
print(f"Train saved    : {len(train_done)} / {len(train_annotations)}")
print(f"Dev saved      : {len(dev_done)}   / {len(dev_annotations)}")

### Extract keypoints

In [ ]:


import gc

extractor = KeypointExtractor()

# Save to Drive so keypoints survive session reset
PHOENIX_KEYPTS = "/content/drive/MyDrive/DGS_Project/keypoints"
os.makedirs(PHOENIX_KEYPTS, exist_ok=True)

def extract_all(annotations, video_dir, keypoint_dir, split_name):
    done, skipped, missing = 0, 0, 0
    total = len(annotations)

    # Build mp4 index for this split
    split_mp4_index = defaultdict(list)
    for fname in sorted(os.listdir(video_dir)):
        if fname.endswith('.mp4'):
            base = '-'.join(fname.replace('.mp4','').split('-')[:-1])
            split_mp4_index[base].append(fname)

    for i, ann in enumerate(annotations):
        safe_name = ann['name'].replace('/', '_')
        out_path  = os.path.join(keypoint_dir, safe_name + '.npy')

        if os.path.exists(out_path):
            skipped += 1
            continue

        mp4_path = get_video_path(ann['name'], video_dir, split_mp4_index)

        if not mp4_path or not os.path.exists(mp4_path):
            missing += 1
            continue

        # Extract from mp4
        kps = extractor.extract_video(mp4_path)   # (T, 543, 3)

        if kps is not None and len(kps) > 0:
            np.save(out_path, kps)
            done += 1
        else:
            missing += 1

        if (i + 1) % 50 == 0:
            print(f"  [{split_name}] {i+1}/{total} | "
                  f"done={done} skipped={skipped} missing={missing}")
            sys.stdout.flush()

        # Free memory every 100 samples
        if (i + 1) % 100 == 0:
            gc.collect()

    print(f"\n✅ [{split_name}] done={done} skipped={skipped} missing={missing}")
    return done


# Extract train first (largest — ~7096 samples, ~2-3 hours on Colab)
print("🔄 Extracting TRAIN...")
extract_all(train_annotations, TRAIN_VIDEO_DIR, PHOENIX_KEYPTS, "TRAIN")

print("\n🔄 Extracting DEV...")
extract_all(dev_annotations, DEV_VIDEO_DIR, PHOENIX_KEYPTS, "DEV")

print("\n✅ Extraction complete.")

In [ ]:
import shutil
import gc

# Copy dataset if needed
if not os.path.exists(DST_BASE):
    print("Copying dataset...")
    shutil.copytree(SRC_BASE, DST_BASE)

train_annotations = load_phoenix_annotations(TRAIN_ANN)
dev_annotations   = load_phoenix_annotations(DEV_ANN)

train_ds    = PHOENIXGlossDataset(train_annotations, PHOENIX_KEYPTS)
val_ds      = PHOENIXGlossDataset(dev_annotations,   PHOENIX_KEYPTS)
gloss_vocab = build_gloss_vocab(train_ds)
print(f"✅ Vocab: {len(gloss_vocab)}")

def collate_fn(batch):
    body_kps  = torch.stack([b["body_kps"] for b in batch])
    face_kps  = torch.stack([b["face_kps"] for b in batch])
    full_kps  = torch.stack([b["full_kps"] for b in batch])
    glosses   = [b["glosses"] for b in batch]
    max_g     = max(len(g) for g in glosses)
    gloss_ids = torch.zeros(len(batch), max_g, dtype=torch.long)
    for i, g_list in enumerate(glosses):
        for j, g in enumerate(g_list):
            gloss_ids[i, j] = gloss_vocab.get(g, gloss_vocab["<unk>"])
    return {
        "body_kps":  body_kps,
        "face_kps":  face_kps,
        "full_kps":  full_kps,
        "glosses":   glosses,
        "gloss_ids": gloss_ids,
        "name":      [b["name"] for b in batch],
        "text":      [b["text"] for b in batch],
    }

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True,
                          num_workers=2, pin_memory=torch.cuda.is_available(),
                          collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False,
                          num_workers=2, pin_memory=torch.cuda.is_available(),
                          collate_fn=collate_fn)

dual_model = DualStreamSignProducer(gloss_vocab_size=len(gloss_vocab)).to(DEVICE)
params = sum(p.numel() for p in dual_model.parameters())
free, total = torch.cuda.mem_get_info()

print(f"✅ Model   : {params:,} parameters")
print(f"   GPU free: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

# Sanity check
dual_model.eval()
with torch.no_grad():
    dummy_gloss = torch.zeros(2, 10, dtype=torch.long).to(DEVICE)
    dummy_body  = torch.zeros(2, 64, 75*3).to(DEVICE)
    dummy_face  = torch.zeros(2, 64, 468*3).to(DEVICE)
    out = dual_model(dummy_gloss, dummy_body, dummy_face)

free, _ = torch.cuda.mem_get_info()
print(f"✅ Forward pass OK — GPU free after: {free/1e9:.2f} GB")
print(f"   body_kps : {out['body_kps'].shape}")
print(f"   face_kps : {out['face_kps'].shape}")

del out, dummy_gloss, dummy_body, dummy_face
gc.collect()
torch.cuda.empty_cache()
free, _ = torch.cuda.mem_get_info()
print(f"✅ Ready for training — GPU free: {free/1e9:.2f} GB")

In [ ]:
import sys
import json

def train_dual_stream(model, train_loader, val_loader,
                      epochs=50, lr=1e-4, grad_clip=1.0):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = DualStreamLoss()
    best_val_loss = float("inf")
    history = {"train": [], "val": []}

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for batch_idx, batch in enumerate(train_loader):
            body_kps  = batch["body_kps"].to(DEVICE)
            face_kps  = batch["face_kps"].to(DEVICE)
            gloss_ids = batch["gloss_ids"].to(DEVICE)
            B, T = body_kps.shape[:2]
            body_flat = body_kps.reshape(B, T, -1)
            face_flat = face_kps.reshape(B, T, -1)
            body_in = torch.cat([torch.zeros(B,1,75*3,device=DEVICE), body_flat[:,:-1]], dim=1)
            face_in = torch.cat([torch.zeros(B,1,468*3,device=DEVICE), face_flat[:,:-1]], dim=1)
            pred = model(gloss_ids, body_in, face_in)
            counter_tgt = torch.ones(B, T, device=DEVICE)
            loss, loss_dict = criterion(pred, body_flat, face_flat, counter_tgt)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            train_losses.append(loss_dict["total"])
            if (batch_idx + 1) % 50 == 0:
                print(f"  Ep {epoch} | Batch {batch_idx+1}/{len(train_loader)} "
                      f"| Loss: {loss_dict['total']:.4f} "
                      f"| Body: {loss_dict['body']:.4f} "
                      f"| Face: {loss_dict['face']:.4f}")
                sys.stdout.flush()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                body_kps  = batch["body_kps"].to(DEVICE)
                face_kps  = batch["face_kps"].to(DEVICE)
                gloss_ids = batch["gloss_ids"].to(DEVICE)
                B, T = body_kps.shape[:2]
                body_flat = body_kps.reshape(B, T, -1)
                face_flat = face_kps.reshape(B, T, -1)
                body_in = torch.cat([torch.zeros(B,1,75*3,device=DEVICE), body_flat[:,:-1]], dim=1)
                face_in = torch.cat([torch.zeros(B,1,468*3,device=DEVICE), face_flat[:,:-1]], dim=1)
                pred = model(gloss_ids, body_in, face_in)
                counter_tgt = torch.ones(B, T, device=DEVICE)
                val_loss, _ = criterion(pred, body_flat, face_flat, counter_tgt)
                val_losses.append(val_loss.item())

        avg_train = np.mean(train_losses)
        avg_val   = np.mean(val_losses)
        scheduler.step()
        history["train"].append(avg_train)
        history["val"].append(avg_val)
        print(f"\nEpoch {epoch:3d}/{epochs} | Train: {avg_train:.4f} | Val: {avg_val:.4f} "
              f"| LR: {scheduler.get_last_lr()[0]:.6f}")
        sys.stdout.flush()

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            ckpt_path = os.path.join(CKPT_DIR, "dual_stream_best.pt")  # ← was missing
            torch.save({
                "epoch":       epoch,
                "model_state": model.state_dict(),
                "optim_state": optimizer.state_dict(),
                "val_loss":    avg_val,
            }, ckpt_path)
            with open(ckpt_path.replace('.pt', '_vocab.json'), 'w') as f:
                json.dump(gloss_vocab, f)
            print(f"  💾 Saved (val={avg_val:.4f}) → {ckpt_path}")
            sys.stdout.flush()

    return model, history   # ← outside for loop, outside if block


# ── Resume from existing checkpoint ─────────────────────────────
ckpt = torch.load(
    os.path.join(CKPT_DIR, "dual_stream_best.pt"),
    map_location=DEVICE,
    weights_only=False
)
dual_model.load_state_dict(ckpt["model_state"])
completed_epochs = ckpt["epoch"]
print(f"✅ Resuming from epoch {completed_epochs} | Val: {ckpt['val_loss']:.4f}")

trained_model, history = train_dual_stream(
    model        = dual_model,
    train_loader = train_loader,
    val_loader   = val_loader,
    epochs       = 50 - completed_epochs,   # ← only remaining epochs
    lr           = 1e-4,
    grad_clip    = 1.0
)

In [ ]:
# Check how many real keypoint files exist
import os
npy_files = [f for f in os.listdir(PHOENIX_KEYPTS) if f.endswith('.npy')]
print(f"Keypoint files extracted : {len(npy_files)}")
print(f"Train samples            : {len(train_ds)}")
print(f"Coverage                 : {len(npy_files)/len(train_ds)*100:.1f}%")

# Check one sample
sample = train_ds[0]
kp = sample['full_kps']
print(f"\nSample keypoints shape : {kp.shape}")
print(f"All zeros?             : {(kp == 0).all().item()}")
print(f"Max value              : {kp.max().item():.4f}")
print(f"Mean nonzero           : {kp[kp!=0].mean().item() if (kp!=0).any() else 'N/A'}")

In [ ]:
# ──  video structure ──────────────────────

sample = train_annotations[0]
print(f"Annotation name : {sample['name']}")
print(f"Gloss           : {sample['gloss']}")
print()

# Check TRAIN_VIDEO_DIR
print(f"TRAIN_VIDEO_DIR : {TRAIN_VIDEO_DIR}")
print(f"Exists          : {os.path.exists(TRAIN_VIDEO_DIR)}")
print()

if os.path.exists(TRAIN_VIDEO_DIR):
    items = sorted(os.listdir(TRAIN_VIDEO_DIR))
    print(f"Total items     : {len(items)}")
    print(f"First 10        : {items[:10]}")
    print()

    # Go one level deeper into first item
    first_item = os.path.join(TRAIN_VIDEO_DIR, items[0])
    if os.path.isdir(first_item):
        sub = sorted(os.listdir(first_item))
        print(f"Inside [{items[0]}]:")
        print(f"  Items : {len(sub)}")
        print(f"  First : {sub[:5]}")

        # Go one more level if needed
        first_sub = os.path.join(first_item, sub[0])
        if os.path.isdir(first_sub):
            subsub = sorted(os.listdir(first_sub))
            print(f"\n  Inside [{sub[0]}]:")
            print(f"    Items : {len(subsub)}")
            print(f"    First : {subsub[:5]}")
    else:
        print(f"First item is a FILE: {first_item}")

print()

# Check if annotation name maps to anything
candidate1 = os.path.join(TRAIN_VIDEO_DIR, sample['name'])
candidate2 = os.path.join(TRAIN_VIDEO_DIR, sample['name'].split('/')[-1])
candidate3 = TRAIN_VIDEO_DIR.replace('train', '') + sample['name']

print(f"Candidate 1 (full name)      : {candidate1}")
print(f"  exists: {os.path.exists(candidate1)}")
print()
print(f"Candidate 2 (last part only) : {candidate2}")
print(f"  exists: {os.path.exists(candidate2)}")
print()
print(f"Candidate 3 (no split prefix): {candidate3}")
print(f"  exists: {os.path.exists(candidate3)}")

### CELL 9: Three.js Avatar Renderer

In [ ]:
# ── CELL 7: Train DE→Gloss Model ────────────────────────────────
from transformers import BertTokenizer
import sys
import transformers, huggingface_hub, tokenizers

def train_de_gloss(model, tokenizer, train_annotations, gloss_vocab,
                   epochs=20, lr=2e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    model.train()

    for epoch in range(1, epochs + 1):
        losses = []
        for ann in train_annotations:
            text = ann.get('text', '')
            if not text:
                continue

            glosses   = ann['gloss'].split()
            gloss_ids = [gloss_vocab.get(g, gloss_vocab['<unk>'])
                         for g in glosses[:50]]
            gloss_ids = [gloss_vocab['<sos>']] + gloss_ids + [gloss_vocab['<eos>']]

            enc = tokenizer(
                text, return_tensors='pt',
                truncation=True, max_length=128
            ).to(DEVICE)

            tgt_in  = torch.tensor([gloss_ids[:-1]], device=DEVICE)
            tgt_out = torch.tensor([gloss_ids[1:]],  device=DEVICE)

            logits = model(
                enc['input_ids'],
                enc['attention_mask'],
                tgt_in
            )

            loss = criterion(
                logits.reshape(-1, logits.shape[-1]),
                tgt_out.reshape(-1)
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        avg = np.mean(losses)
        print(f"Epoch {epoch}/{epochs} | Loss: {avg:.4f}")
        sys.stdout.flush()

    # Save checkpoint
    ckpt_path = os.path.join(CKPT_DIR, "de_gloss_model.pt")
    torch.save(model.state_dict(), ckpt_path)
    print(f"✅ DE→Gloss saved → {ckpt_path}")
    return model


# Initialize
de_gloss_tokenizer = BertTokenizer.from_pretrained("deepset/gbert-base")
de_gloss_model     = DEGlossModel(vocab_size=len(gloss_vocab)).to(DEVICE)

# Load checkpoint if exists, else train
de_gloss_ckpt = os.path.join(CKPT_DIR, "de_gloss_model.pt")
if os.path.exists(de_gloss_ckpt):
    de_gloss_model.load_state_dict(
        torch.load(de_gloss_ckpt, map_location=DEVICE, weights_only=False)
    )
    print("✅ DE→Gloss checkpoint loaded — skipping training")
else:
    print("🔄 Training DE→Gloss model (~2-3 hours)...")
    de_gloss_model = train_de_gloss(
        model             = de_gloss_model,
        tokenizer         = de_gloss_tokenizer,
        train_annotations = train_annotations,
        gloss_vocab       = gloss_vocab,
        epochs            = 20,
        lr                = 2e-4
    )

In [ ]:
import os

# ── Delete bad checkpoint ────────────────────────────────────────
ckpt_path = os.path.join(CKPT_DIR, "de_gloss_model.pt")
os.remove(ckpt_path)
print("🗑️  Deleted bad checkpoint")

# ── Retrain from scratch ─────────────────────────────────────────
de_gloss_model = DEGlossModel(vocab_size=len(gloss_vocab)).to(DEVICE)

de_gloss_model = train_de_gloss(
    model             = de_gloss_model,
    tokenizer         = de_gloss_tokenizer,
    train_annotations = train_annotations,
    gloss_vocab       = gloss_vocab,
    epochs            = 20,
    lr                = 2e-4
)

# ── Test after training ──────────────────────────────────────────
de_gloss_model.eval()
test_sentences = [
    "Heute ist es kalt.",
    "Guten Morgen, wie ist das Wetter?",
    "Es schneit morgen.",
]
for sent in test_sentences:
    glosses = de_gloss_model.generate(
        sent, de_gloss_tokenizer, gloss_vocab, device=DEVICE
    )
    print(f"  {sent}")
    print(f"  → {glosses}")
    print()

In [ ]:
# ── Check what's in the checkpoint ──────────────────────────────
import os
ckpt_path = os.path.join(CKPT_DIR, "de_gloss_model.pt")

ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
print(f"Checkpoint keys: {list(ckpt.keys()) if isinstance(ckpt, dict) else 'state_dict'}")

# Check if it was actually trained by looking at weights
de_gloss_model.load_state_dict(
    ckpt if not isinstance(ckpt, dict) else ckpt
)

# Test generate
german  = "Heute ist es kalt."
glosses = de_gloss_model.generate(
    german, de_gloss_tokenizer, gloss_vocab, device=DEVICE
)
print(f"Generated glosses: {glosses}")

In [ ]:
import transformers, huggingface_hub, tokenizers
print(f"transformers  : {transformers.__version__}")
print(f"huggingface_hub: {huggingface_hub.__version__}")
print(f"tokenizers    : {tokenizers.__version__}")

### DE→Gloss using Helsinki-NLP

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL: Lightweight DE→Gloss using Helsinki-NLP (small, fits GPU)
# ════════════════════════════════════════════════════════════════
from transformers import MarianMTModel, MarianTokenizer
import torch, os, sys, numpy as np
from torch.utils.data import DataLoader, Dataset

class MarianGlossModel:
    """
    Helsinki-NLP MarianMT (de→en) repurposed for DE→Gloss.
    ~300MB — fits easily in 14.5GB GPU.
    """
    def __init__(self, gloss_vocab, device='cpu'):
        self.device      = device
        self.gloss_vocab = gloss_vocab

        print("⏳ Loading Helsinki-NLP/opus-mt-de-en ...")
        self.tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-de-en")
        self.model     = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-de-en").to(device)

        # Add gloss tokens to tokenizer vocab
        new_tokens = [g for g in gloss_vocab if g not in self.tokenizer.get_vocab()]
        if new_tokens:
            self.tokenizer.add_tokens(new_tokens)
            self.model.resize_token_embeddings(len(self.tokenizer))
            print(f"  ➕ Added {len(new_tokens)} gloss tokens to tokenizer")

        print(f"✅ MarianMT loaded — {sum(p.numel() for p in self.model.parameters())/1e6:.0f}M params")

    def fine_tune(self, train_annotations, epochs=10, batch_size=32, lr=3e-4,
                  ckpt_path=None, grad_accum=2):

        class GlossDataset(Dataset):
            def __init__(self, annotations, tokenizer):
                self.samples = [
                    (a['text'], a['gloss'])
                    for a in annotations
                    if a.get('text') and a.get('gloss')
                ]
                self.tok = tokenizer

            def __len__(self): return len(self.samples)

            def __getitem__(self, idx):
                src, tgt = self.samples[idx]

                src_enc = self.tok(
                    src, max_length=96, truncation=True,
                    padding='max_length', return_tensors='pt'
                )
                tgt_enc = self.tok(
                    tgt, max_length=48, truncation=True,
                    padding='max_length', return_tensors='pt'
                )
                labels = tgt_enc['input_ids'].squeeze().clone()
                labels[labels == self.tok.pad_token_id] = -100

                return {
                    'input_ids'      : src_enc['input_ids'].squeeze(),
                    'attention_mask' : src_enc['attention_mask'].squeeze(),
                    'labels'         : labels
                }

        dataset    = GlossDataset(train_annotations, self.tokenizer)
        dataloader = DataLoader(dataset, batch_size=batch_size,
                                shuffle=True, pin_memory=True)

        optimizer  = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=0.01)
        scheduler  = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=lr,
            steps_per_epoch=len(dataloader) // grad_accum,
            epochs=epochs
        )

        best_loss  = float('inf')
        self.model.train()

        for epoch in range(1, epochs + 1):
            losses = []
            optimizer.zero_grad()

            for step, batch in enumerate(dataloader):
                input_ids      = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels         = batch['labels'].to(self.device)

                out  = self.model(input_ids      = input_ids,
                                  attention_mask = attention_mask,
                                  labels         = labels)
                loss = out.loss / grad_accum
                loss.backward()
                losses.append(loss.item() * grad_accum)

                # Gradient accumulation
                if (step + 1) % grad_accum == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

            avg = np.mean(losses)
            print(f"Epoch {epoch:02d}/{epochs} | Loss: {avg:.4f}")
            sys.stdout.flush()

            if avg < best_loss:
                best_loss = avg
                if ckpt_path:
                    self.model.save_pretrained(ckpt_path)
                    self.tokenizer.save_pretrained(ckpt_path)

        print(f"✅ Done. Best loss: {best_loss:.4f} → {ckpt_path}")

    def predict(self, german_text, num_beams=5, max_new_tokens=40):
        self.model.eval()
        with torch.no_grad():
            enc = self.tokenizer(
                german_text, return_tensors='pt',
                truncation=True, max_length=96
            ).to(self.device)

            out_ids = self.model.generate(
                **enc,
                num_beams            = num_beams,
                max_new_tokens       = max_new_tokens,
                early_stopping       = True,
                no_repeat_ngram_size = 2,
            )
            decoded = self.tokenizer.decode(out_ids[0], skip_special_tokens=True)

        # Filter to known gloss vocab
        raw     = decoded.upper().split()
        glosses = [g for g in raw if g in self.gloss_vocab]
        return glosses if glosses else raw


# ── Initialize ───────────────────────────────────────────────────
MARIAN_CKPT = os.path.join(CKPT_DIR, "marian_gloss_finetuned")

# Free memory from previous model
try:
    del mbart_gloss
    torch.cuda.empty_cache()
    print("🗑️  Freed mBART from GPU memory")
except: pass

marian_gloss = MarianGlossModel(gloss_vocab, device=DEVICE)

if os.path.exists(MARIAN_CKPT) and os.path.exists(os.path.join(MARIAN_CKPT, "config.json")):
    print("✅ Loading fine-tuned checkpoint...")
    marian_gloss.model     = MarianMTModel.from_pretrained(MARIAN_CKPT).to(DEVICE)
    marian_gloss.tokenizer = MarianTokenizer.from_pretrained(MARIAN_CKPT)
    print("✅ Checkpoint loaded")
else:
    print("🔄 Fine-tuning MarianMT (~10-15 min)...")
    marian_gloss.fine_tune(
        train_annotations = train_annotations,
        epochs            = 10,
        batch_size        = 32,
        lr                = 3e-4,
        ckpt_path         = MARIAN_CKPT,
        grad_accum        = 2
    )



In [ ]:
import os, re, numpy as np
from transformers import MarianMTModel, MarianTokenizer
print("✅ imports ok")

In [ ]:
DE_STOP = {
    'ist', 'sind', 'war', 'waren', 'bin', 'bist',
    'der', 'die', 'das', 'ein', 'eine', 'einen',
    'und', 'oder', 'aber', 'weil', 'wenn', 'dass',
    'ich', 'du', 'er', 'sie', 'es', 'wir', 'ihr',
    'mir', 'dir', 'ihm', 'ihr', 'uns', 'euch',
    'nicht', 'kein', 'keine', 'auch', 'noch', 'schon',
    'sehr', 'zu', 'an', 'in', 'auf', 'mit', 'von',
    'für', 'bei', 'nach', 'aus', 'über', 'unter',
    'haben', 'hat', 'hatte', 'werden', 'wird', 'wurde',
    'this', 'is', 'the', 'a', 'an', 'and', 'or'
}

In [ ]:
def run_pipeline_marian(english_text, keypoint_dir):
    print("=" * 55)
    print(f"  INPUT : {english_text}")
    print("=" * 55)

    german_text = translate_en_to_de(english_text)
    print(f"🇩🇪 German  : {german_text}")

    glosses = marian_gloss.predict(german_text, num_beams=5)
    print(f"📋 Glosses  : {glosses}")

    # Keyword fallback if empty or all unknown
    if not glosses or not any(g in gloss_vocab for g in glosses):
        kw = de_keyword_to_glosses(german_text, gloss_vocab)
        if kw:
            print(f"🔑 Keyword fallback : {kw}")
            glosses = kw

    trimmed_kps = render_real_signing(" ".join(glosses), gloss_vocab, keypoint_dir)
    return trimmed_kps


# ── Test ─────────────────────────────────────────────────────────
run_pipeline_marian("weather is warm", PHOENIX_KEYPTS);


In [ ]:
import gc, torch
free  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n📊 GPU: {free:.1f}GB used / {total:.1f}GB total")
print(f"📊 Free: {total - free:.1f}GB")

### Example Usage

In [ ]:
MIXAMO_AVATAR_HTML = """
<!DOCTYPE html>
<html>
<head>
<style>
  * { margin:0; padding:0; box-sizing:border-box; }
  body { background:#0f0f1a; font-family:monospace; }
  #canvas-container { width:100%; height:500px; position:relative; }
  #controls {
    position:absolute; bottom:12px; left:50%; transform:translateX(-50%);
    display:flex; gap:12px; align-items:center;
    background:rgba(0,0,0,0.6); padding:8px 16px; border-radius:24px;
  }
  button {
    background:#6c63ff; color:white; border:none;
    padding:6px 14px; border-radius:12px; cursor:pointer;
  }
  #gloss-display {
    position:absolute; top:12px; left:50%; transform:translateX(-50%);
    background:rgba(0,0,0,0.7); color:#6c63ff; padding:6px 16px;
    border-radius:12px; font-size:14px; letter-spacing:2px;
  }
  #frame-label { color:#aaa; font-size:12px; }
  #loading {
    position:absolute; top:50%; left:50%; transform:translate(-50%,-50%);
    color:white; font-size:18px;
  }
</style>
</head>
<body>
<div id="canvas-container">
  <div id="loading">Loading avatar...</div>
  <div id="gloss-display">LOADING...</div>
  <div id="controls">
    <button onclick="prevFrame()">\u25C0</button>
    <button onclick="togglePlay()">\u25B6 Play</button>
    <button onclick="nextFrame()">\u25B6</button>
    <span id="frame-label">Frame 0 / 0</span>
  </div>
</div>

<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script src="https://cdn.jsdelivr.net/gh/mrdoob/three.js@r128/examples/js/loaders/GLTFLoader.js"></script>

<script>
const KEYPOINTS = KEYPOINTS_PLACEHOLDER;
const GLOSSES   = GLOSSES_PLACEHOLDER;
const FPS = 8;
const T   = KEYPOINTS.length;

const container = document.getElementById("canvas-container");
const scene     = new THREE.Scene();
scene.background = new THREE.Color(0x0f0f1a);

const camera = new THREE.PerspectiveCamera(60, container.clientWidth/500, 0.1, 5000);
camera.position.set(0, 150, 350);
camera.lookAt(0, 100, 0);

const renderer = new THREE.WebGLRenderer({ antialias:true });
renderer.setSize(container.clientWidth, 500);
container.insertBefore(renderer.domElement, container.firstChild);

scene.add(new THREE.AmbientLight(0xffffff, 0.9));
const dir = new THREE.DirectionalLight(0xffffff, 0.8);
dir.position.set(2, 4, 3);
scene.add(dir);

const ground = new THREE.Mesh(
  new THREE.PlaneGeometry(600, 600),
  new THREE.MeshLambertMaterial({ color:0x1a1a2e })
);
ground.rotation.x = -Math.PI/2;
scene.add(ground);

let bones = {};
let avatar = null;
const restAxis = {};
let currentFrame = 0;
let playing      = false;
let lastTime     = 0;

function renderLoop() {
  requestAnimationFrame(renderLoop);
  renderer.render(scene, camera);
}
renderLoop();

const loader = new THREE.GLTFLoader();
loader.load(
  'AVATAR_URL_PLACEHOLDER',
  (gltf) => {
    avatar = gltf.scene;
    avatar.scale.set(100, 100, 100);
    avatar.position.set(0, 0, 0);
    scene.add(avatar);

    const box = new THREE.Box3().setFromObject(avatar);
    avatar.position.y = -box.min.y;

    avatar.traverse((obj) => {
      if (obj.isBone || obj.type === 'Bone') {
        bones[obj.name] = obj;
      }
    });

    console.log("Bones found:", Object.keys(bones));
    captureRestAxes();
    document.getElementById("loading").style.display = "none";

    if (T > 0) {
      updateFrame(0);
      playing = true;
      document.querySelector("#controls button:nth-child(2)").textContent = "\u23F8 Pause";
      requestAnimationFrame(animate);
    }
  },
  (progress) => {
    if (progress.total) {
      const pct = Math.round(progress.loaded / progress.total * 100);
      document.getElementById("loading").textContent = `Loading avatar... ${pct}%`;
    }
  },
  (error) => {
    document.getElementById("loading").textContent = "Failed to load avatar";
    console.error(error);
  }
);

// Capture each bone's rest "tail" direction in its OWN local frame, using
// the next bone in the chain as the reference. setFromUnitVectors then
// rotates this stored axis to the live target direction (in parent-local
// frame). This is what fixes the "limb only twists" bug.
function captureRestAxes() {
  const chain = {
    'mixamorigSpine':        'mixamorigSpine1',
    'mixamorigSpine1':       'mixamorigSpine2',
    'mixamorigSpine2':       'mixamorigNeck',
    'mixamorigNeck':         'mixamorigHead',
    'mixamorigLeftArm':      'mixamorigLeftForeArm',
    'mixamorigLeftForeArm':  'mixamorigLeftHand',
    'mixamorigLeftHand':     'mixamorigLeftHandMiddle1',
    'mixamorigRightArm':     'mixamorigRightForeArm',
    'mixamorigRightForeArm': 'mixamorigRightHand',
    'mixamorigRightHand':    'mixamorigRightHandMiddle1'
  };
  let captured = 0, missingChild = [];
  for (const par in chain) {
    const child = chain[par];
    if (bones[par] && bones[child]) {
      restAxis[par] = bones[child].position.clone().normalize();
      captured++;
    } else if (bones[par]) {
      restAxis[par] = new THREE.Vector3(0, 1, 0);
      missingChild.push(par);
    }
  }
  console.log("Captured rest axes for", captured, "bones");
  if (missingChild.length) console.warn("Fallback +Y for:", missingChild);
}

function kpToWorld(kp) {
  const x = kp[0], y = kp[1], z = kp[2];
  return new THREE.Vector3((x - 0.5) * 300, (0.75 - y) * 400, -z * 150);
}
function isZero(kp) {
  return kp[0] === 0 && kp[1] === 0 && kp[2] === 0;
}

function setBoneFromDir(boneName, dirWorld) {
  const bone = bones[boneName];
  const axis = restAxis[boneName];
  if (!bone || !axis || !dirWorld || dirWorld.lengthSq() < 1e-6) return;

  const parentWorldQuat = new THREE.Quaternion();
  if (bone.parent) bone.parent.getWorldQuaternion(parentWorldQuat);
  const dirLocal = dirWorld.clone().applyQuaternion(parentWorldQuat.invert()).normalize();
  bone.quaternion.setFromUnitVectors(axis, dirLocal);
}

function updateBones(kps) {
  if (!avatar || Object.keys(restAxis).length === 0) return;

  const get = (i) => isZero(kps[i]) ? null : kpToWorld(kps[i]);
  const sub = (a, b) => (a && b) ? new THREE.Vector3().subVectors(b, a) : null;

  const lSh = get(11), rSh = get(12);
  const lEl = get(13), rEl = get(14);
  const lWr = get(15), rWr = get(16);
  const lHi = get(23), rHi = get(24);

  // Hand-middle-MCP in your stored layout (body 0-32 + leftHand 33-53 + rightHand 54-74):
  // hand idx 9 (middle MCP) -> 42 (left), 63 (right). Falls back to wrist if missing.
  const lHand = get(42) || lWr;
  const rHand = get(63) || rWr;

  if (lHi && rHi && lSh && rSh) {
    const hipMid  = lHi.clone().add(rHi).multiplyScalar(0.5);
    const shoMid  = lSh.clone().add(rSh).multiplyScalar(0.5);
    const spineUp = new THREE.Vector3().subVectors(shoMid, hipMid);
    setBoneFromDir('mixamorigSpine',  spineUp);
    setBoneFromDir('mixamorigSpine1', spineUp);
    setBoneFromDir('mixamorigSpine2', spineUp);
  }

  setBoneFromDir('mixamorigLeftArm',      sub(lSh, lEl));
  setBoneFromDir('mixamorigLeftForeArm',  sub(lEl, lWr));
  setBoneFromDir('mixamorigLeftHand',     sub(lWr, lHand));
  setBoneFromDir('mixamorigRightArm',     sub(rSh, rEl));
  setBoneFromDir('mixamorigRightForeArm', sub(rEl, rWr));
  setBoneFromDir('mixamorigRightHand',    sub(rWr, rHand));
}

function updateFrame(f) {
  if (T === 0) return;
  const kps = KEYPOINTS[f];
  updateBones(kps);

  const gi = Math.floor((f / Math.max(T,1)) * GLOSSES.length);
  document.getElementById("gloss-display").textContent =
    GLOSSES[Math.min(gi, Math.max(GLOSSES.length-1, 0))] || "";
  document.getElementById("frame-label").textContent = `Frame ${f+1} / ${T}`;
  currentFrame = f;
}

function togglePlay() {
  playing = !playing;
  document.querySelector("#controls button:nth-child(2)").textContent =
    playing ? "\u23F8 Pause" : "\u25B6 Play";
  if (playing) { lastTime = 0; requestAnimationFrame(animate); }
}
function prevFrame() { playing=false; updateFrame(Math.max(0, currentFrame-1)); }
function nextFrame() { playing=false; updateFrame(Math.min(T-1, currentFrame+1)); }

function animate(time) {
  if (!playing) return;
  if (time - lastTime > 1000/FPS) {
    lastTime = time;
    const next = currentFrame + 1;
    if (next >= T) {
      playing = false;
      document.querySelector("#controls button:nth-child(2)").textContent = "\u25B6 Play";
      updateFrame(0);
      return;
    }
    updateFrame(next);
  }
  requestAnimationFrame(animate);
}

window.addEventListener("resize", () => {
  camera.aspect = container.clientWidth/500;
  camera.updateProjectionMatrix();
  renderer.setSize(container.clientWidth, 500);
});
</script>
</body>
</html>
"""


def render_mixamo_avatar(keypoints, glosses, glb_path="/content/avatar.glb"):
    import json
    import base64
    from IPython.display import display, HTML
    from google.colab import output

    with open(glb_path, 'rb') as f:
        glb_data = f.read()
    glb_b64    = base64.b64encode(glb_data).decode('utf-8')
    avatar_url = f"data:model/gltf-binary;base64,{glb_b64}"

    kp_list = keypoints.tolist()
    html    = MIXAMO_AVATAR_HTML
    html    = html.replace("KEYPOINTS_PLACEHOLDER", json.dumps(kp_list))
    html    = html.replace("GLOSSES_PLACEHOLDER",   json.dumps(glosses))
    html    = html.replace("AVATAR_URL_PLACEHOLDER", avatar_url)

    output.eval_js("0")
    display(HTML(html))
    print(f"Mixamo avatar rendered: {len(keypoints)} frames | {glosses}")


In [ ]:
def render_real_signing(gloss_text, gloss_vocab, keypoint_dir):
    target_glosses = gloss_text.upper().split()

    best_match = None
    best_score = -1

    for ann in train_annotations:
        ann_glosses = ann['gloss'].upper().split()
        score = sum(1 for g in target_glosses if g in ann_glosses)
        if score > best_score:
            best_score  = score
            best_match  = ann

    if best_match is None:
        print("❌ No match found")
        return

    safe_name = best_match['name'].replace('/', '_')
    kp_path   = os.path.join(keypoint_dir, safe_name + '.npy')

    if not os.path.exists(kp_path):
        print(f"❌ Keypoint file not found: {kp_path}")
        return

    kps         = np.load(kp_path)
    all_glosses = best_match['gloss'].split()

    matched_glosses = [g for g in all_glosses if g.upper() in target_glosses]
    if not matched_glosses:
        matched_glosses = target_glosses

    T           = kps.shape[0]
    n_all       = len(all_glosses)
    frames_each = T // n_all if n_all > 0 else T

    matched_frames = []
    for i, g in enumerate(all_glosses):
        if g.upper() in target_glosses:
            start = max(0, i * frames_each - 5)         # 5 frames before
            end   = min((i + 1) * frames_each + 5, T)   # 5 frames after
            matched_frames.append(kps[start:end])

    trimmed_kps = np.concatenate(matched_frames, axis=0) if matched_frames else kps

    print(f"✅ Matched glosses : {matched_glosses}")
    print(f"   Frames          : {trimmed_kps.shape[0]} (trimmed from {T})")

    # ── REPLACE old stick figure with Mixamo avatar ──
    render_mixamo_avatar(trimmed_kps, matched_glosses, glb_path="/content/avatar.glb")

    return trimmed_kps

In [ ]:
def run_pipeline_marian(english_text, keypoint_dir):
    print("=" * 55)
    print(f"  INPUT : {english_text}")
    print("=" * 55)

    german_text = translate_en_to_de(english_text)
    print(f"🇩🇪 German  : {german_text}")

    glosses = marian_gloss.predict(german_text, num_beams=5)
    print(f"📋 Glosses  : {glosses}")

    # Keyword fallback if empty or all unknown
    if not glosses or not any(g in gloss_vocab for g in glosses):
        kw = de_keyword_to_glosses(german_text, gloss_vocab)
        if kw:
            print(f"🔑 Keyword fallback : {kw}")
            glosses = kw

    trimmed_kps = render_real_signing(" ".join(glosses), gloss_vocab, keypoint_dir)

# ── Test ─────────────────────────────────────────────────────────
run_pipeline_marian("Weather is warm", PHOENIX_KEYPTS);